# Autoencoder Waveforms Training (Google Colab)
Notebook for training the CNN Autoencoder for waveforms, optimized for Google Colab with GPU acceleration.

### Prerequisites on Colab:
1. Ensure the runtime is set to **GPU** (*Runtime > Change runtime type > T4 GPU*).
2. The file `nonhyb_noprec_dataset_q8.hdf` must be placed in your Google Drive root directory (`/content/drive/MyDrive/`).


### How to Mount Google Drive in Google Colab
To load large datasets and preserve model checkpoints across disconnects, mount your Google Drive:

1. **Code execution**: Run the setup cell below. It executes:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```
2. **Grant permission**: A prompt will appear asking for authorization; click **"Connect to Google Drive"** and choose your Google account.
3. **Alternative GUI method**: Click the folder icon on the left sidebar in Colab, then click the **"Mount Drive"** button.
4. **Directory access**: Your files will appear under `/content/drive/MyDrive/`.


In [ ]:
# 1. GPU Check and Google Drive Mount
import os
import shutil
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Enable GPU via Runtime > Change runtime type")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_hdf = '/content/drive/MyDrive/nonhyb_noprec_dataset_q8.hdf'
    local_hdf = 'nonhyb_noprec_dataset_q8.hdf'
    
    if os.path.exists(drive_hdf):
        if not os.path.exists(local_hdf):
            print("Copying dataset from Drive to Colab local disk at high speed...")
            shutil.copy(drive_hdf, local_hdf)
            print("Copy completed successfully!")
        else:
            print("Dataset already present on local disk.")
    else:
        print(f"WARNING: {drive_hdf} not found! Check the path on Google Drive.")
except ImportError:
    print("Local environment (not Colab). Ensure 'nonhyb_noprec_dataset_q8.hdf' is in the current directory.")


### 2. Helper Modules (`models.py`, `utils.py`, `train.py`)
Automatic generation of project files in the Colab workspace if loaded as a standalone notebook.


In [ ]:
%%writefile models.py
import torch
import torch.nn as nn


class Encoder1(nn.Module):
    def __init__(self, negative_slope: float = 0.2):
        '''
        Docstring: 
        Convolutional layer bc they are leaner than FC layer and they keep temporal info better
        Batchnorm to stabilize training
        last layer before latent space is a fully connected, slope to be optuned
        Number of params: 3.5e4
        Number of params of fc AE: 3e6
        '''
        super().__init__()
        self.c1 = nn.Conv1d(in_channels= 1, out_channels=16, kernel_size=2, stride=2, padding=0, bias=False) #out lenght= 1024
        self.bn1 = nn.BatchNorm1d(num_features=16)
        self.c2 = nn.Conv1d(in_channels= 16, out_channels=12, kernel_size=4, stride=4, padding=0, bias=False) # o.l = 256
        self.bn2 = nn.BatchNorm1d(num_features=12)
        self.c3 = nn.Conv1d(in_channels=12, out_channels=8, kernel_size=4, stride=4, padding=0, bias=False) # ol = 64
        self.bn3 = nn.BatchNorm1d(num_features=8) 
        self.fc = nn.Linear(in_features=512, out_features=64, bias = False)
        self.act = nn.LeakyReLU(negative_slope=negative_slope)

    def forward(self,x):
        x = self.act(self.bn1(self.c1(x)))
        x = self.act(self.bn2(self.c2(x)))
        x = self.act(self.bn3(self.c3(x)))
        return self.fc(torch.flatten(x, start_dim=1))


class Decoder1(nn.Module):
    def __init__(self, negative_slope: float = 0.2):
        super().__init__()
        self.ct1 = nn.ConvTranspose1d(in_channels=1, out_channels=16, kernel_size=4, stride=4, padding=0, bias=False) #ol = 256
        self.bn1 = nn.BatchNorm1d(num_features=16)
        self.ct2 = nn.ConvTranspose1d(in_channels=16, out_channels=12, kernel_size=4, stride=4, padding=0, bias=False) # ol = 1024
        self.bn2 = nn.BatchNorm1d(num_features=12)
        self.ct3 = nn.ConvTranspose1d(in_channels=12, out_channels=8, kernel_size=2, stride=2, padding=0, bias=False) #ol = 2048
        self.bn3 = nn.BatchNorm1d(num_features=8)
        self.ct4 = nn.Conv1d(in_channels=8, out_channels=1, kernel_size=7, stride=1, padding=3, bias=False)
        self.act = nn.LeakyReLU(negative_slope=negative_slope)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.act(self.bn1(self.ct1(x)))
        x = self.act(self.bn2(self.ct2(x)))
        x = self.act(self.bn3(self.ct3(x)))
        return self.ct4(x)


class Autoencoder_CNN_1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder1()
        self.decoder = Decoder1()
    def forward(self,x):
        return self.decoder(self.encoder(x))


In [ ]:
%%writefile utils.py
import numpy as np
import torch
import torch.nn as nn


def separate_phase_ampl(dataset):
    '''
    Separates phase and amplitude in 2 distinct datasets
    Inputs:
        dataset: complex dataset
    Outputs:
        ampl: amplitude dataset
        phase: phase dataset
    '''

    ampl = abs(dataset)
    phase_non = -1*np.unwrap(np.angle(dataset))/100
    out = phase_non
    phase= out-out[..., np.newaxis,0]
    return [ampl, phase]



def count_model_params(model: nn.Module) -> dict:
    """Computes trainable, non-trainable, and total parameters in PyTorch."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return {
        "trainable": trainable,
        "non_trainable": non_trainable,
        "total": trainable + non_trainable,
    }

def get_conv1d_params(in_length: int, out_length: int, in_channels: int, out_channels: int, bias: bool = True):
    """
    Finds valid (kernel_size, stride, padding) configurations to map in_length -> out_length,
    along with total learnable parameters and an instantiated nn.Conv1d layer.
    """
    candidates = []
    for stride in [1, 2, 4, 8]:
        for kernel_size in range(2, 17):
            for padding in range(kernel_size):
                num = in_length + 2 * padding - kernel_size
                if num >= 0 and num % stride == 0:
                    l_out = num // stride + 1
                    if l_out == out_length:
                        num_params = out_channels * in_channels * kernel_size + (out_channels if bias else 0)
                        candidates.append({
                            "kernel_size": kernel_size,
                            "stride": stride,
                            "padding": padding,
                            "trainable_params": num_params,
                            "layer": nn.Conv1d(
                                in_channels=in_channels,
                                out_channels=out_channels,
                                kernel_size=kernel_size,
                                stride=stride,
                                padding=padding,
                                bias=bias,
                            ),
                        })
    return candidates


def get_conv_transpose1d_params(
    in_length: int,
    out_length: int,
    in_channels: int,
    out_channels: int,
    bias: bool = True,
):
    """
    Finds valid (kernel_size, stride, padding) configurations to map in_length -> out_length,
    along with total learnable parameters and an instantiated nn.ConvTranspose1d layer.
    """
    candidates = []
    for stride in [1, 2, 4, 8]:
        for kernel_size in range(2, 17):
            for padding in range(kernel_size):
                l_out = (in_length - 1) * stride - 2 * padding + kernel_size
                if l_out == out_length:
                    num_params = in_channels * out_channels * kernel_size + (out_channels if bias else 0)
                    candidates.append({
                        "kernel_size": kernel_size,
                        "stride": stride,
                        "padding": padding,
                        "trainable_params": num_params,
                        "layer": nn.ConvTranspose1d(
                            in_channels=in_channels,
                            out_channels=out_channels,
                            kernel_size=kernel_size,
                            stride=stride,
                            padding=padding,
                            bias=bias,
                        ),
                    })
    return candidates


def train_one_epoch(model, optimizer, loss_fn, device, epoch_index, tb_writer, loader):
    model.train(True)
    running_loss = 0.0
    last_loss = 0.0

    for i, batch in enumerate(loader):
        inputs = batch.float().to(device)
        if inputs.ndim == 2:
            inputs = inputs.unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, inputs)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 100
            print(f"  batch {i + 1} loss: {last_loss:.6f}")
            tb_x = epoch_index * len(loader) + i + 1
            tb_writer.add_scalar("Loss/train", last_loss, tb_x)
            running_loss = 0.0

    return last_loss


In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from utils import separate_phase_ampl
from torch import Tensor
from torch.utils.data import random_split, TensorDataset
import h5py
import numpy as np
from torch.optim import Adam
from utils import train_one_epoch
from torch.utils.tensorboard import SummaryWriter
from models import Autoencoder_CNN_1
from datetime import datetime


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## Load dataset from hdf

with h5py.File("nonhyb_noprec_dataset_q8.hdf", "r") as f:
    waveforms_ds = f["Waveforms"]
    n_samples, sample_len = waveforms_ds.shape
    amplitude = np.empty((n_samples, sample_len), dtype=np.float32)
    chunk_size = 10000
    for start in range(0, n_samples, chunk_size):
        end = min(start + chunk_size, n_samples)
        amplitude[start:end] = np.abs(waveforms_ds[start:end]).astype(np.float32)

ampl_tensor = torch.from_numpy(amplitude)

# split in test, train and validation for both
a_train, a_val, a_test = random_split(ampl_tensor, [0.8,0.1,0.1], generator=torch.Generator().manual_seed(42))
#p_train, p_val, p_test = random_split(splitted_waveforms[1], [0.8,0.1,0.1], generator=torch.Generator().manual_seed(42))

# 1 dataloader for each one
a_train_loader = DataLoader(a_train, batch_size = 128, shuffle= True)
a_val_loader = DataLoader(a_val, batch_size=256, shuffle=False)
a_test_loader = DataLoader(a_test, batch_size=256, shuffle=False)
#p_train_loader = DataLoader(p_train, batch_size = 128, shuffle= True)
#p_val_loader = DataLoader(p_val, batch_size=256, shuffle=False)
#p_test_loader = DataLoader(p_test, batch_size=256, shuffle=False)


model1 = Autoencoder_CNN_1().to(device)

## Loss function
loss_fn = nn.L1Loss(reduction='mean')

# Optimizer function
opt_fn = Adam(model1.parameters(), lr=1e-3, weight_decay=0)

training_loader = a_train_loader
validation_loader = a_val_loader

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
writer = SummaryWriter(f"runs/AE_CNN_1_Amplitude_{timestamp}")
EPOCHS = 10
best_vloss = float("inf")

for epoch in range(EPOCHS):
    print(f"EPOCH {epoch + 1}:")
    avg_loss = train_one_epoch(model1, opt_fn, loss_fn, device, epoch, writer, training_loader)

    running_vloss = 0.0
    model1.eval()

    with torch.no_grad():
        for i, vbatch in enumerate(validation_loader):
            vinputs = vbatch.float().to(device)
            if vinputs.ndim == 2:
                vinputs = vinputs.unsqueeze(1)
            voutputs = model1(vinputs)
            vloss = loss_fn(voutputs, vinputs)
            running_vloss += vloss.item()

    avg_vloss = running_vloss / (i + 1)
    print(f"LOSS train: {avg_loss:.6f} | valid: {avg_vloss:.6f}")

    writer.add_scalars(
        "Training and Validation Loss",
        {"Training": avg_loss, "Validation": avg_vloss},
        epoch + 1,
    )
    writer.flush()

    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = f"model_{timestamp}_{epoch}.pt"
        torch.save(model1.state_dict(), model_path)


### 4. Memory-Optimized Dataset Loading and DataLoader Creation
Stream waveforms from HDF5 in chunks directly into `float32` (800 MB RAM total) to prevent Google Colab OOM crashes, then construct GPU DataLoaders.


In [ ]:
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split

# 1. Stream amplitude in float32 chunks (avoids 11.8 GB RAM peak)
h5_path = "nonhyb_noprec_dataset_q8.hdf"
with h5py.File(h5_path, "r") as f:
    waveforms_ds = f["Waveforms"]
    n_samples = len(waveforms_ds)
    sample_len = waveforms_ds.shape[1]
    print(f"HDF5 dataset: {n_samples} samples of length {sample_len}")
    
    # Preallocate float32 array: only 800 MB RAM for all 102,400 samples
    amplitude = np.empty((n_samples, sample_len), dtype=np.float32)
    chunk_size = 10000
    for start in range(0, n_samples, chunk_size):
        end = min(start + chunk_size, n_samples)
        amplitude[start:end] = np.abs(waveforms_ds[start:end]).astype(np.float32)

print(f"Amplitude loaded: {amplitude.shape} (RAM usage: {amplitude.nbytes / (1024**2):.1f} MB)")

# 2. Convert to PyTorch Tensor & split (80% train, 10% validation, 10% test)
ampl_tensor = torch.from_numpy(amplitude)
generator = torch.Generator().manual_seed(42)
a_train, a_val, a_test = random_split(ampl_tensor, [0.8, 0.1, 0.1], generator=generator)

# 3. GPU DataLoaders
pin = torch.cuda.is_available()
a_train_loader = DataLoader(a_train, batch_size=128, shuffle=True, pin_memory=pin, num_workers=2)
a_val_loader = DataLoader(a_val, batch_size=256, shuffle=False, pin_memory=pin, num_workers=2)
a_test_loader = DataLoader(a_test, batch_size=256, shuffle=False)

print(f"Train batches: {len(a_train_loader)} | Val batches: {len(a_val_loader)} | Test batches: {len(a_test_loader)}")


### 5. Model Definition, Loss, and Optimizer
Instantiate the CNN Autoencoder from `models.py` and configure the Adam optimizer.


In [ ]:
from models import Autoencoder_CNN_1
from utils import count_model_params
from torch.optim import Adam
import torch.nn as nn

# Instantiate model on GPU
model = Autoencoder_CNN_1().to(device)
params_info = count_model_params(model)
print(f"Model parameters: {params_info}")

# Reconstruction loss (L1) and Optimizer
loss_fn = nn.L1Loss(reduction='mean')
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=0)


### 6. Training Loop and Google Drive Checkpointing
Train the model for the specified number of epochs. Best checkpoints are saved both locally and to Google Drive.


In [ ]:
import os
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
from utils import train_one_epoch

# Backup folder on Google Drive to prevent losing weights if Colab disconnects
drive_ckpt_dir = "/content/drive/MyDrive/AE_checkpoints"
os.makedirs(drive_ckpt_dir, exist_ok=True)
os.makedirs("runs", exist_ok=True)

training_loader = a_train_loader
validation_loader = a_val_loader

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
writer = SummaryWriter(f"runs/AE_CNN_1_Amplitude_{timestamp}")
EPOCHS = 10
best_vloss = float("inf")

for epoch in range(EPOCHS):
    print(f"\n=== EPOCH {epoch + 1}/{EPOCHS} ===")
    
    # Training with helper function from utils.py
    avg_loss = train_one_epoch(model, optimizer, loss_fn, device, epoch, writer, training_loader)

    # Validation
    model.eval()
    running_vloss = 0.0
    with torch.no_grad():
        for i, vbatch in enumerate(validation_loader):
            vinputs = vbatch.float().to(device)
            if vinputs.ndim == 2:
                vinputs = vinputs.unsqueeze(1)
            voutputs = model(vinputs)
            vloss = loss_fn(voutputs, vinputs)
            running_vloss += vloss.item()

    avg_vloss = running_vloss / (i + 1)
    print(f"Epoch {epoch + 1} | Train Loss: {avg_loss:.6f} | Valid Loss: {avg_vloss:.6f}")

    writer.add_scalars(
        "Training and Validation Loss",
        {"Training": avg_loss, "Validation": avg_vloss},
        epoch + 1,
    )
    writer.flush()

    # Save best model checkpoint
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        ckpt_filename = f"model_best_epoch_{epoch + 1}_{timestamp}.pt"
        local_path = os.path.join("runs", ckpt_filename)
        torch.save(model.state_dict(), local_path)
        
        # Backup copy to Drive if available
        if os.path.exists(drive_ckpt_dir):
            drive_path = os.path.join(drive_ckpt_dir, ckpt_filename)
            torch.save(model.state_dict(), drive_path)
            print(f"[*] New best model saved to Drive: {drive_path}")
        else:
            print(f"[*] New best model saved locally: {local_path}")

print(f"\nTraining complete! Best Validation Loss: {best_vloss:.6f}")


### 7. Visual Evaluation of Reconstructions
Graphical comparison between original waveforms and Autoencoder reconstructions on test set samples.


In [ ]:
import matplotlib.pyplot as plt

model.eval()
test_batch = next(iter(a_test_loader))
test_inputs = test_batch.float().to(device)
if test_inputs.ndim == 2:
    test_inputs = test_inputs.unsqueeze(1)

with torch.no_grad():
    reconstructions = model(test_inputs)

# Plot first 3 samples
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for idx in range(3):
    orig = test_inputs[idx, 0].cpu().numpy()
    recon = reconstructions[idx, 0].cpu().numpy()
    
    axes[idx].plot(orig, label="Original", color="blue", alpha=0.7)
    axes[idx].plot(recon, label="Reconstructed", color="red", linestyle="--", alpha=0.8)
    axes[idx].set_title(f"Sample {idx + 1} (Amplitude)")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.xlabel("Sample index")
plt.tight_layout()
plt.show()
